## Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


## Membuat SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


## Membaca Dataset

In [3]:
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
print("Total baris:", df.count())
df.show(5)

Total baris: 2500


+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



## A. Feature Engineering

In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# StringIndexer untuk kolom nama_menu, cabang, dan waktu_pesan
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

model_menu = indexer_menu.fit(df)
df_indexed = model_menu.transform(df)

model_cabang = indexer_cabang.fit(df_indexed)
df_indexed = model_cabang.transform(df_indexed)

model_waktu = indexer_waktu.fit(df_indexed)
df_indexed = model_waktu.transform(df_indexed)

df_indexed.select("nama_menu", "menu_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_idx").show(5)

# VectorAssembler seluruh fitur
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+--------------+--------+----------+----------+-----------+---------+
|     nama_menu|menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_idx|
+--------------+--------+----------+----------+-----------+---------+
|Kentang Goreng|     5.0|  Semarang|       2.0|      Malam|      1.0|
|Kentang Goreng|     5.0|Yogyakarta|       1.0|       Sore|      3.0|
|Kentang Goreng|     5.0|  Semarang|       2.0|      Siang|      2.0|
|Kentang Goreng|     5.0|Yogyakarta|       1.0|       Pagi|      0.0|
|    Roti Bakar|     4.0|  Magelang|       0.0|      Malam|      1.0|
+--------------+--------+----------+----------+-----------+---------+
only showing top 5 rows



+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



## B. Train/Test Split & Pelatihan Model

In [5]:
from pyspark.ml.regression import LinearRegression

# Split data 75% training dan 25% testing (seed=101)
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

# Pelatihan model
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("\nModel berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839


Jumlah data testing  : 661



Model berhasil dilatih!
Koefisien tiap fitur: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


## C. Evaluasi Model

In [6]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)
predictions.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "total_penjualan", "prediction").show(10)

# Menghitung RMSE dan R2
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f} ({r2*100:.1f}%)")

+---------+--------+-----------+---+------------+------------------+------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+-----------+---+------------+------------------+------------------+
|Americano|Magelang|      Malam|  1|       25000|19721.170446051627|31056.146105506836|
|Americano|Magelang|      Malam|  1|       25000|  21236.0956379149|31056.146105506836|
|Americano|Magelang|      Malam|  4|       22000|  85834.8564401191| 88127.37331410211|
|Americano|Magelang|      Malam|  4|       22000|102174.09100317833| 88127.37331410211|
|Americano|Magelang|      Malam|  4|       30000|113044.63431555482|112310.74025938749|
|Americano|Magelang|      Malam|  5|       22000|115810.29600214415|110174.03658512786|
|Americano|Magelang|      Malam|  5|       25000|140386.40273615732|119242.79918960987|
|Americano|Magelang|       Pagi|  2|       30000| 73842.64420790318| 67877.77492201277|
|Americano|Magelang|       Pagi|

RMSE : Rp 10,901
R²   : 0.9107 (91.1%)


Menurut saya model ini sudah cukup baik untuk dipakai memprediksi omzet karena nilai R² mencapai 0.9107 (sekitar 91%), yang artinya sebagian besar variasi data penjualan berhasil dijelaskan oleh model. Nilai RMSE sekitar Rp 10.901 juga tergolong wajar dan relatif kecil terhadap rentang total penjualan.

## D. Eksperimen Perbandingan Fitur

In [7]:
# VectorAssembler baru hanya dengan fitur qty dan harga_satuan
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)

df_sederhana = assembler_sederhana.transform(df_indexed)
train_sederhana, test_sederhana = df_sederhana.randomSplit([0.75, 0.25], seed=101)

lr_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model_sederhana = lr_sederhana.fit(train_sederhana)

predictions_sederhana = model_sederhana.transform(test_sederhana)

rmse_sederhana = evaluator_rmse.evaluate(predictions_sederhana)
r2_sederhana = evaluator_r2.evaluate(predictions_sederhana)

print(f"Model Utama (5 Fitur) -> RMSE: Rp {rmse:,.0f} | R²: {r2:.4f}")
print(f"Model Kedua (2 Fitur) -> RMSE: Rp {rmse_sederhana:,.0f} | R²: {r2_sederhana:.4f}")

Model Utama (5 Fitur) -> RMSE: Rp 10,901 | R²: 0.9107
Model Kedua (2 Fitur) -> RMSE: Rp 10,896 | R²: 0.9108


Kesimpulannya, penambahan informasi menu, cabang, dan waktu tidak meningkatkan akurasi prediksi secara berarti karena nilai R² kedua model hampir sama persis (0.9107 vs 0.9108). Hal ini wajar karena total penjualan pada dasarnya hanya dipengaruhi langsung oleh perkalian jumlah unit (qty) dan harga satuan produk.

## E. Prediksi Skenario Baru

In [8]:
# 3 skenario pesanan baru
skenario_data = [
    ("Kopi Susu", "Magelang", "Pagi", 3, 20000),
    ("Cappuccino", "Yogyakarta", "Sore", 2, 25000),
    ("Roti Bakar", "Semarang", "Malam", 4, 18000)
]
kolom = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_skenario = spark.createDataFrame(skenario_data, schema=kolom)

# Menggunakan model StringIndexer yang sudah di-fit sebelumnya
df_skenario_indexed = model_menu.transform(df_skenario)
df_skenario_indexed = model_cabang.transform(df_skenario_indexed)
df_skenario_indexed = model_waktu.transform(df_skenario_indexed)

# Menggunakan VectorAssembler yang sama
df_skenario_final = assembler.transform(df_skenario_indexed)

# Prediksi dengan model utama
prediksi_skenario = model.transform(df_skenario_final)
prediksi_skenario.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |3  |20000       |59572.01525544525|
|Cappuccino|Yogyakarta|Sore       |2  |25000       |53814.94297982256|
|Roti Bakar|Semarang  |Malam      |4  |18000       |76101.40186674664|
+----------+----------+-----------+---+------------+-----------------+



## Penutup

In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
